# Transformando Audio a Imágenes para Redes Neuronales Convolucionales (CNN)

¡Bienvenidos al módulo de Procesamiento de Audio!

En esta sesión aprenderemos cómo las máquinas "ven" el sonido. Para que una Arquitectura Convolucional (CNN) pueda procesar audio de la manera más efectiva posible, la estrategia dominante en Deep Learning es convertir las ondas de sonido unidimensionales (1D) en representaciones gráficas bidimensionales (2D) llamadas **Espectrogramas**.

A lo largo de este notebook utilizaremos las abstracciones de alto nivel de PyTorch (`torchaudio.transforms`) para el preprocesamiento, lo cual nos evitará programar algoritmos matemáticos complejos desde cero.

In [1]:
import torch
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
import urllib.request
import os
from IPython.display import Audio, display
import ipywidgets as widgets
from ipywidgets import interact

# Configuramos el tamaño y resolución estándar para nuestras gráficas de matplotlib
plt.rcParams['figure.figsize'] = [12, 4]
plt.rcParams['figure.dpi'] = 100

## Sección 1: Carga y Visualización 1D (La Forma de Onda)

Todo sonido digital es fundamentalmente una secuencia larguísima de números (amplitudes) registrados a través del tiempo.

Para comenzar, descargaremos un archivo de sonido en formato `.wav`. Al cargarlo en `torchaudio`, obtenemos dos cosas:
1. **Waveform (Tensor):** El arreglo de amplitudes en formato `[canales, tiempo]`.
2. **Sample Rate (Frecuencia de Muestreo):** Cuántas muestras se tomaron por segundo (ej. 44100 Hz = 44100 valores por segundo).

In [ ]:
# 1. Usamos urllib para descargar un audio de prueba desde los tutoriales de PyTorch
url = "https://raw.githubusercontent.com/pytorch/audio/main/test/assets/tutorial-assets/steam-train-whistle-daniel_simon.wav"
filename = "steam_train.wav"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

# 2. Cargar el audio usando la abstracción de PyTorch
waveform, sample_rate = torchaudio.load(filename)

print(f"Forma del Tensor de Audio (Waveform): {waveform.shape}")
print(f"Frecuencia de muestreo (Sample Rate): {sample_rate} Hz")

# 3. Reproducimos el clip para escuchar qué estamos analizando
display(Audio(waveform.numpy()[0], rate=sample_rate))

# 4. Graficar la onda acústica contra el tiempo (1D)
tiempo = torch.arange(waveform.shape[1]) / sample_rate  # Convertir muestras a segundos

plt.figure(figsize=(10, 4))
plt.plot(tiempo, waveform[0].numpy(), color='royalblue')
plt.title("Visualización 1D: Forma de Onda Original del Audio")
plt.xlabel("Tiempo (s)")
plt.ylabel("Amplitud")
plt.grid(True, alpha=0.3)
plt.show()

## Sección 2: Del Tiempo a la Frecuencia (Espectrograma - STFT)

Para una red neuronal (y para nosotros), ver esa onda enredada 1D hace que sea muy difícil distinguir si se está tocando una nota Do, Fa, si habla un hombre o una mujer, etc.

La solución es aplicar la **Transformada Creciente de Fourier a Corto Plazo (STFT)**. En esencia, divide todo el audio en "ventanas" cortas de tiempo, y calcula las frecuencias presentes en cada ventana.

El nivel de detalle visual está dominado por dos parámetros:
- `n_fft` (Tamaño de la Ventana Fourier): Dicta la cantidad de información temporal que se procesa a la vez. Al aumentarlo (ej: 2048), la resolución de frecuencia mejora (se ven líneas horizontales muy finas) pero la resolución de tiempo se vuelve borrosa.
- `hop_length`: Es el "salto" que da la ventana. Afecta directamente cuántos pixeles de *ancho* tendrá la imagen resultante.

*(¡Juega con los sliders a continuación para observar el compromiso entre tiempo y frecuencia!)*

In [ ]:
@interact(
    n_fft=widgets.Dropdown(options=[256, 512, 1024, 2048], value=1024, description='n_fft:'),
    hop_length=widgets.IntSlider(min=64, max=1024, step=64, value=512, description='hop_length:')
)
def interact_spectrogram(n_fft, hop_length):
    # Instanciamos la abstracción de PyTorch que realiza la transformada por nosotros.
    # power=2.0 significa que nos retorna el Espectrograma de Potencia (Magnitud al cuadrado).
    spec_transform = T.Spectrogram(
        n_fft=n_fft,
        hop_length=hop_length,
        power=2.0
    )
    
    # Pasamos nuestro tensor a través de la transformación
    spectrogram = spec_transform(waveform)
    
    # Para mejorar el contraste meramente al visualizar (usamos log para evitar valores que dominen la gráfica visual)
    spectrogram_plot = torch.log(spectrogram + 1e-9)
    
    plt.figure(figsize=(10, 4))
    # Origin lower invierte el eje Y para que frecuencias bajas estén abajo, "aspect='auto'" adapta la grid a la celda
    plt.imshow(spectrogram_plot[0].numpy(), origin='lower', aspect='auto', cmap='magma')
    plt.title(f"Espectrograma STFT | Dimensiones del Tensor: {spectrogram.shape}")
    plt.xlabel("Frames (Resolución de Tiempo)")
    plt.ylabel("Bins (Resolución de Frecuencia)")
    plt.colorbar()
    plt.show()

## Sección 3: La Perspectiva Humana (Mel Spectrogram)

El espectrograma anterior reparte las diferentes frecuencias de manera lineal. Sin embargo, el oído humano no percibe los sonidos matemáticamente lineales. Si subimos un tono de 100Hz a 200Hz, notamos una subida abrupta. Pero si subimos de 10,000Hz a 10,100Hz apenas y notamos diferencia.

El **Espectrograma de Mel** agrupa la parte superior del espectro y concentra la resolución en las bajas frecuencias. Desde una perspectiva de Ciencias de Datos, esta es de hecho una técnica brutal de **Reducción de Dimensionalidad**. Si configuramos `n_mels = 64`, estamos compactando de quizás cientos de frecuencias a usar únicamente 64 renglones ("Filtros Mel"). Esto ayuda a la CNN a enfocarse en lo importante y a procesar infinitamente más rápido.

In [ ]:
@interact(
    n_mels=widgets.IntSlider(min=10, max=128, step=10, value=64, description='n_mels (Filtros):')
)
def interact_mel_spectrogram(n_mels):
    # T.MelSpectrogram abstrae STFT y la aplicación de Banco de Filtros Mel.
    mel_transform = T.MelSpectrogram(
        sample_rate=sample_rate,
        n_fft=1024,
        hop_length=512,
        n_mels=n_mels  # Modificamos la cantidad de canales/dimensiones en el eje Y
    )
    
    mel_spectrogram = mel_transform(waveform)
    mel_spectrogram_plot = torch.log(mel_spectrogram + 1e-9)
    
    plt.figure(figsize=(10, 4))
    plt.imshow(mel_spectrogram_plot[0].numpy(), origin='lower', aspect='auto', cmap='viridis')
    plt.title(f"Espectrograma Mel | Ajuste a {n_mels} Filtros (Tensor Forma: {mel_spectrogram.shape})")
    plt.xlabel("Frames (Tiempo)")
    plt.ylabel(f"Bancos de Frecuencia ({n_mels} Mels)")
    plt.colorbar()
    plt.show()

## Sección 4: Decibeles (La Escala Logarítmica Final)

Al igual que las frecuencias, la **amplitud (volumen)** también es mal percibida de forma puramente lineal por un humano. Usualmente la traducimos a **Decibeles (dB)**, que crecen logarítmicamente.

Como último paso en el preprocesamiento de Pytorch (previo a inyectar imágenes en nuestro DataLoader de la red), transformamos el tensor de magnitud a Decibeles con `AmplitudeToDB`. Esta representación final equilibra totalmente la matriz y se comporta estadísticamente espectacular dentro de una CNN (¡sus gradientes no van a explotar!).

In [ ]:
# 1. Definimos el Pipeline completo de preprocesamiento, configuraciones estándar:
mel_spec_transform = T.MelSpectrogram(
    sample_rate=sample_rate,
    n_fft=1024,
    hop_length=512,
    n_mels=64
)

# 2. Transformador final que convierte el espectrograma (puro) de potencia a matriz en Decibeles (dB)
amplitude_to_db = T.AmplitudeToDB(stype='power', top_db=80.0)

# Ejecutamos las dos transformaciones en serie al tensor del audio
mel_spec = mel_spec_transform(waveform)
mel_spec_db = amplitude_to_db(mel_spec)

# Confirmamos la dimensionalidad para la red neuronal:
print("="*50)
print("TENSOR FINAL - LISTO PARA INTRODUCIR A LA CNN")
print(f"Shape original del objeto: {mel_spec_db.shape}")
print(f"  -> {mel_spec_db.shape[0]} canal de color (Nuestra 'Imagen Blanca y Negra')")
print(f"  -> {mel_spec_db.shape[1]} Pixeles de alto (Reducido a {mel_spec_db.shape[1]} variables/filtros Mel)")
print(f"  -> {mel_spec_db.shape[2]} Pixeles de ancho (Número total de pasos en tiempo/frames)")
print("="*50)

# Graficado final demostrativo de la 'imagen'
plt.figure(figsize=(10, 4))
plt.imshow(mel_spec_db[0].numpy(), origin='lower', aspect='auto', cmap='Blues')
plt.title("Entrada CNN: Tensor Final en Decibeles (Mel-Spectrogram dB)")
plt.xlabel("Tiempo [Frames]")
plt.ylabel("Frecuencias [Filtros Mel]")
plt.colorbar(format="%+2.0f dB")
plt.show()